# Assignment 1
Classification using KNN

In [ ]:
# Import standard libraries
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import recall_score, precision_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.datasets import load_wine

# Load the Wine dataset
wine_data = load_wine()

# Convert to DataFrame
wine_df = pd.DataFrame(wine_data.data, columns=wine_data.feature_names)

# Bind the 'class' (wine target) to the DataFrame
wine_df['class'] = wine_data.target

# Display the DataFrame
wine_df

## Question 1: Data Inspection

In [ ]:
# (i) How many observations (rows) does the dataset contain?
print(wine_df.shape[0])

In [ ]:
# (ii) How many variables (columns) does the dataset contain?
print(wine_df.shape[1])

In [ ]:
# (iii) Variable type and unique values of 'class'
print(wine_df['class'].dtype)
print(wine_df['class'].unique())

In [ ]:
# (iv) Number of predictor variables (all columns except 'class')
print(len(wine_df.columns) - 1)

## Question 2: Standardization and Data Splitting

In [ ]:
# Select predictors (excluding the last column)
predictors = wine_df.iloc[:, :-1]

# Standardize the predictors
scaler = StandardScaler()
predictors_standardized = pd.DataFrame(scaler.fit_transform(predictors), columns=predictors.columns)

# Display the head of the standardized predictors
print(predictors_standardized.head())

**(i)** Standardization is important for KNN because the algorithm calculates distances between data points. If features are on very different scales, the feature with the largest scale will dominate the distance calculation, biasing the model. Standardizing puts all features on the same scale so each contributes equally.

**(ii)** We do not standardize the response variable `class` because it is a categorical label we are trying to predict, not a numeric input feature. Transforming it would distort its meaning.

**(iii)** Setting a random seed ensures reproducibility — anyone running the code will get the same train/test split and results. The specific value of the seed does not matter; what matters is that one is set consistently.

In [ ]:
# (iii) Set a random seed for reproducibility
np.random.seed(123)

# (iv) Split the data into training (75%) and testing (25%) sets
X_train, X_test, y_train, y_test = train_test_split(
    predictors_standardized,
    wine_df['class'],
    test_size=0.25,
    random_state=123
)

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)

## Question 3: Model Initialization and Cross-Validation

In [ ]:
# Initialize the KNN classifier
knn = KNeighborsClassifier()

# Define the parameter grid for n_neighbors (1 to 50)
param_grid = {'n_neighbors': list(range(1, 51))}

# Set up GridSearchCV with 10-fold cross-validation
grid_search = GridSearchCV(knn, param_grid, cv=10, scoring='accuracy')

# Fit on training data
grid_search.fit(X_train, y_train)

# Best n_neighbors
print("Best n_neighbors:", grid_search.best_params_['n_neighbors'])

## Question 4: Model Evaluation

In [ ]:
# Fit KNN with the best n_neighbors found from grid search
best_k = grid_search.best_params_['n_neighbors']
best_knn = KNeighborsClassifier(n_neighbors=best_k)
best_knn.fit(X_train, y_train)

# Predict on the test set
y_pred = best_knn.predict(X_test)

# Evaluate with accuracy
print("Best n_neighbors:", best_k)
print("Test Accuracy:", accuracy_score(y_test, y_pred))